In [0]:
%sql

CREATE OR REPLACE TABLE DEV.CLAIMS_PROJECT.gold_claim_lag AS
SELECT
    ClaimId,
    PatientId,
    ProviderId,
    ProviderSpecialty,
    ProviderLocation,
    DiagnosisCode,
    ProcedureCode,
    ClaimType,
    ClaimStatus,
    ClaimDate,
    claim_received_date,
    paid_date,
    ClaimAmount,
    claim_lag_days,
    payment_lag_days,

    CASE 
        WHEN claim_lag_days BETWEEN 0 AND 7 THEN '0-7 days'
        WHEN claim_lag_days BETWEEN 8 AND 30 THEN '8-30 days'
        WHEN claim_lag_days BETWEEN 31 AND 60 THEN '31-60 days'
        WHEN claim_lag_days BETWEEN 61 AND 90 THEN '61-90 days'
        WHEN claim_lag_days > 90 THEN '90+ days'
        ELSE 'Invalid lag'
    END AS claim_lag_bucket,

    CASE
        WHEN paid_date <= ClaimDate + INTERVAL 30 DAYS THEN 'Paid within 30 days'
        WHEN paid_date <= ClaimDate + INTERVAL 60 DAYS THEN 'Paid within 60 days'
        WHEN paid_date > ClaimDate + INTERVAL 60 DAYS THEN 'Late Payment'
        ELSE 'Invalid'
    END AS payment_timeliness_status

FROM dev.claims_project.silver_claims_enriched;

In [0]:
%sql
SELECT *
FROM dev.claims_project.gold_claim_lag
LIMIT 20;

In [0]:
%sql
SELECT CLAIM_LAG_DAYS, COUNT(*) AS TOTAL_CLAIMS FROM dev.claims_project.gold_claim_lag GROUP BY CLAIM_LAG_DAYS ORDER BY total_claims DESC

In [0]:
%sql
SELECT
    payment_timeliness_status,
    COUNT(*) AS total_claims
FROM dev.claims_project.gold_claim_lag
GROUP BY payment_timeliness_status;